In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download("ai12-level1-project")

print("Path to competition files:", path)

/Users/joelchoi/study/projects/project1_3team/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to competition files: /Users/joelchoi/.cache/kagglehub/competitions/ai12-level1-project


https://api.fda.gov/drug/event.json?limit=1

In [ ]:
import json
import shutil
import torch
import yaml


from pathlib import Path
from collections import defaultdict
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

Matplotlib is building the font cache; this may take a moment.


In [28]:
SRC = Path.home() / path / "sprint_ai_project1_data"
DST = Path("data/pill_yolo")

SRC_TRAIN_IMAGES = SRC / "train_images"
SRC_TEST_IMAGES = SRC / "test_images"
SRC_ANNOTATIONS = SRC / "train_annotations"

SUPER_KEY = ord("*")
torch.manual_seed(SUPER_KEY)
g = torch.Generator().manual_seed(SUPER_KEY)

test_size = 0.2

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)
# fid_device = torch.device("cpu")

### 전체 category_id 수집 -> 0부터 연속 정수로 매핑

In [ ]:
cat_map = {}  # {원본 category_id: yolo_class_id}
cat_names = {}  # {yolo_class_id: 약품명}

all_jsons = sorted(SRC_ANNOTATIONS.rglob("*.json"))

print("jsons len", len(all_jsons))
for json_file in all_jsons:
    data = json.load(open(json_file))
    for cat in data["categories"]:
        if cat["id"] not in cat_map:
            idx = len(cat_map)
            cat_map[cat["id"]] = idx
            cat_names[idx] = cat["name"]

print(f"클래스 수: {len(cat_map)}")


jsons len 763
클래스 수: 56


### 이미지별로 annotation 모으기

In [ ]:
# key: 파일명(str), value: list of (class_id, bbox, img_w, img_h)
img_annotations = defaultdict(list)

for json_file in all_jsons:
    data = json.load(open(json_file))
    img_info = data["images"][0]
    file_name = img_info["file_name"]
    img_w, img_h = img_info["width"], img_info["height"]

    for ann in data["annotations"]:
        original_cat_id = ann["category_id"]
        yolo_cls = cat_map[original_cat_id]
        x_min, y_min, w, h = ann["bbox"]

        x_center = (x_min + w / 2) / img_w
        y_center = (y_min + h / 2) / img_h
        norm_w = w / img_w
        norm_h = h / img_h

        img_annotations[file_name].append((
            yolo_cls,
            x_center,
            y_center,
            norm_w,
            norm_h,
        ))

print(f"어노테이션 있는 이미지 수: {len(img_annotations)}")

어노테이션 있는 이미지 수: 232


### train val 8:2

In [ ]:
image_names = sorted(img_annotations.keys())
train_names, val_names = train_test_split(
    image_names, test_size=test_size, random_state=SUPER_KEY
)
print(f"train: {len(train_names)}, val: {len(val_names)}")

train: 185, val: 47


### YOLO COCO용 DIR 생성 + 파일 복사

In [ ]:
for split, names in [("train", train_names), ("val", val_names)]:
    image_dir = DST / "images" / split
    label_dir = DST / "labels" / split
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    for name in names:
        # copy images
        shutil.copy2(SRC_TRAIN_IMAGES / name, image_dir / name)

        # label txt
        txt_name = Path(name).stem + ".txt"
        with open(label_dir / txt_name, "w") as f:
            for (
                cls,
                xc,
                yc,
                w,
                h,
            ) in img_annotations[name]:
                f.write(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

# copy test images
test_dir = DST / "images" / "test"
test_dir.mkdir(parents=True, exist_ok=True)
for img_path in sorted(SRC_TEST_IMAGES.iterdir()):
    shutil.copy2(img_path, test_dir / img_path.name)

print("완료! 디렉토리 구조:")
for p in sorted(DST.rglob("*")):
    if p.is_dir():
        count = len(list(p.iterdir()))
        print(f"  {p.relative_to(DST)}/  ({count} files)")

완료! 디렉토리 구조:
  images/  (3 files)
  images/test/  (842 files)
  images/train/  (185 files)
  images/val/  (47 files)
  labels/  (2 files)
  labels/train/  (185 files)
  labels/val/  (47 files)


### YOLO를 위한 yaml 파일 생성

In [32]:
data_yaml = {
    "path": str(DST.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(cat_map),
    "names": cat_names,  # {0: "약품명", 1: "약품명", ...}
}

yaml_path = DST / "data.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, allow_unicode=True, default_flow_style=False)

print(f"저장: {yaml_path}")

저장: data/pill_yolo/data.yaml


### 학습

In [ ]:
yolo_model = YOLO("yolo11n.pt")

results = yolo_model.train(
    data=str(DST / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    project=str(Path("outputs") / "joel/yolo11n"),
    name="pill_yolo11n",
)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/pill_yolo/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pill_yolo11n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, proj

### 추론

In [ ]:
inf_yolo11_model = YOLO("runs/detect/outputs/joel/yolo11n/pill_yolo11n/weights/best.pt")

results = inf_yolo11_model.predict(
    source=str(DST / "images" / "test"),
    imgsz=640,
    save=True,
    project=str(Path("outputs") / "joel"),
    name="pill_yolo11n_predict",
)

image 1/842 /Users/joelchoi/study/projects/project1_3team/data/pill_yolo/images/test/1.png: 640x512 1 보령부스파정 5mg, 1 울트라셋이알서방정, 1 로수젯정10/5밀리그램, 41.8ms
image 2/842 /Users/joelchoi/study/projects/project1_3team/data/pill_yolo/images/test/10.png: 640x512 1 보령부스파정 5mg, 1 가바토파정 100mg, 1 기넥신에프정(은행엽엑스)(수출용), 30.9ms
image 3/842 /Users/joelchoi/study/projects/project1_3team/data/pill_yolo/images/test/100.png: 640x512 1 보령부스파정 5mg, 1 가바토파정 100mg, 1 신바로정, 1 리바로정 4mg, 34.5ms
image 4/842 /Users/joelchoi/study/projects/project1_3team/data/pill_yolo/images/test/1003.png: 640x512 1 리피토정 20mg, 1 기넥신에프정(은행엽엑스)(수출용), 1 엑스포지정 5/160mg, 1 제미메트서방정 50/1000mg, 1 트윈스타정 40/5mg, 33.1ms
image 5/842 /Users/joelchoi/study/projects/project1_3team/data/pill_yolo/images/test/1004.png: 640x512 1 리피토정 20mg, 1 기넥신에프정(은행엽엑스)(수출용), 1 제미메트서방정 50/1000mg, 1 트윈스타정 40/5mg, 31.6ms
image 6/842 /Users/joelchoi/study/projects/project1_3team/data/pill_yolo/images/test/1005.png: 640x512 1 리피토정 20mg, 1 기넥신에프정(은행엽엑스)(수출용), 1 제미메트서방정 50/1

In [ ]:
metrics = inf_yolo11_model.val(data=yaml_path)

ap_matrix = metrics.box.all_ap  # (num_classes, 10)

# IoU 0.75~0.95 = index 5~9
ap_75_95 = ap_matrix[:, 5:]  # (num_classes, 5)
map_75_95 = ap_75_95.mean()  # 전체 mAP@[0.75:0.95]
per_class = ap_75_95.mean(axis=1)  # 클래스별 mAP@[0.75:0.95]

print(f"mAP@[0.50:0.95]: {metrics.box.map:.4f}")
print(f"mAP@[0.75:0.95]: {map_75_95:.4f}")
print(f"mAP@0.50:        {metrics.box.map50:.4f}")
print(f"mAP@0.75:        {metrics.box.map75:.4f}")

Ultralytics 8.4.80 🚀 Python-3.14.6 torch-2.12.1 CPU (Apple M2 Max)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7424.9±1865.2 MB/s, size: 1788.5 KB)
val: Scanning /Users/joelchoi/study/projects/project1_3team/data/pill_yolo/labels/val.cache... 47 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 47/47 21.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.6s/it 4.7s4.0ss
                   all         47        152      0.859      0.667      0.909       0.88
            보령부스파정 5mg          1          1      0.652          1      0.995      0.895
           가바토파정 100mg          2          2      0.727        0.5      0.662      0.662
              놀텍정 10mg          1          1          1          0      0.995      0.895
         비모보정 500/20mg          1          1      0.954          1      0.995      0.895
           뮤테란캡슐 100mg          4          4      0.805          1      0.995      0.932
           